# Using resources (Files)

A notebook recipe for working with Files (resources) on the Istari Digital Platform.

Uses **istari-digital-client** — `V3Client` (V3 resources API) where available, otherwise the v2 `Client`.

Reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` from [`samples/.env`](../.env).

### Platform UI terms (used in each step)

- **Files page** — the main list of uploaded resources in the Istari web application (menu item is often labeled **Files**).
- **File detail page** — the page for a single resource after you click a row; it shows metadata and tabs such as **Versions**, **Comments**, and **Share**.
- **`UI:` link** — a full URL printed at the end of many code cells; open it in your browser to view that resource.

### Actions demonstrated

- Upload a resource from your device (with and without `display_name`)
- Upload with external identifier and version label
- Search, filter, and get resources by id, revision, and external identifiers; resolve current user for owner filter; browse with cursor pagination
- Add a comment on a resource
- View resource details, versions, comments, and the list of users the resource is shared with (and their roles)
- Upload a new version, compare revisions, download content
- Archive a throwaway resource (main demo resources stay active for UI review)
- Cleanup: archive every resource created in this run (final cell)

### Prerequisites

- **`istari-digital-client`** and **`python-dotenv`** (from cookbook root: `uv sync --group dev`).
- In [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`.

### Install kernel (optional)

From the cookbook repository root:

```bash
uv sync --group dev
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Select **Python (istari-client-cookbook)** in the kernel picker.

### Running cells individually

Run **Setup**, then the demo cells in order. **Setup** only connects clients and creates `state`. **§1** defines `section`, `track`, `verify_resource_metadata`, and `report_upload` (reused in §2–3 and §11). Later cells add their own small helpers where needed. Cells reuse variables from earlier steps when present; otherwise set `resource_id = PLACEHOLDER_ID` (or let the cell derive values from the platform).

### Cleanup

After inspecting the platform UI, run the final **Cleanup** cell in the **same kernel session** to archive every resource created during this run and remove the temp download file.

Resource ids are tracked in the in-memory `state` dict (no file on disk). If the kernel was restarted, set `state` manually in the cleanup cell (see its standalone comment) or re-run the demo cells from the top.


## Setup

Client connection, sample paths, and `state` for cleanup. Helper functions are defined in the first cell that needs each one (§1 upload, §4 search, §7 access).

In [ ]:
import os
import re
import tempfile
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
from pathlib import Path

import dotenv
from istari_digital_client import Configuration
from istari_digital_client.client import Client
from istari_digital_client.v3_client import V3Client

EXPECTED_CLIENT_VERSION = "10.10.0"
RECIPE_TAG = "using-resources-recipe"
PLACEHOLDER_ID = "REPLACE_WITH_RESOURCE_ID"

_cwd = Path.cwd()
SAMPLES_DIR = _cwd.parent if (_cwd.parent / ".env").exists() else _cwd / "samples"

dotenv.load_dotenv(SAMPLES_DIR / ".env")
registry_url, token = os.environ.get("ISTARI_REGISTRY_URL"), os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
if not registry_url or not token:
    raise RuntimeError("Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env")

installed = pkg_version("istari-digital-client")
assert installed == EXPECTED_CLIENT_VERSION, (
    f"Expected istari-digital-client=={EXPECTED_CLIENT_VERSION}, got {installed}"
)

config = Configuration(registry_url=registry_url, registry_auth_token=token)
v3, client = V3Client(config), Client(config)

_match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", registry_url)
ui_base = registry_url.rstrip("/") if not _match else f"{_match.group(1)}{_match.group(2)}"

xlsx_v1 = SAMPLES_DIR / "Group3-UAS-Requirements.xlsx"
xlsx_v2 = SAMPLES_DIR / "Group3-UAS-Requirements-v2.xlsx"
if not xlsx_v1.is_file():
    raise FileNotFoundError(f"Sample spreadsheet not found: {xlsx_v1}")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
state: dict = {"run_id": run_id, "resource_ids": [], "comment_ids": []}

print(f"istari-digital-client {EXPECTED_CLIENT_VERSION}")
print(f"Registry: {registry_url}")

## 1 · Upload a resource from your device (no display_name)

### What to check in the web app

After this cell runs, open the **Files** page using the `UI:` link printed in the output. Find the new row: because we did **not** set `display_name`, the list shows the **original filename** from your computer (for example `Group3-UAS-Requirements.xlsx`), not a custom title.

In [ ]:
from istari_digital_client.v3.models import ResourceTypeDto
from istari_digital_client.v3_client import V3Client


def section(title: str) -> None:
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")


def track(resource_id: str) -> None:
    state["resource_ids"].append(resource_id)


def verify_resource_metadata(
    v3: V3Client,
    resource_id: str,
    *,
    name: str,
    display_name: str | None,
    external_identifier: str | None,
    version_name: str | None,
) -> None:
    r = v3.get_resource(resource_id=resource_id)
    assert r.name == name
    assert r.display_name == display_name
    assert r.external_identifier == external_identifier
    assert r.version_name == version_name


def report_upload(
    ui_base: str,
    resource_id: str,
    *,
    name: str,
    display_name: str | None,
    external_identifier: str | None,
    version_name: str | None,
) -> None:
    print(f"Uploaded resource_id={resource_id}")
    print(f"  name={name!r} display_name={display_name!r}")
    print(f"  external_id={external_identifier!r} version={version_name!r}")
    print(f"UI: {ui_base}/files/{resource_id}")


section("1 · Upload a resource from your device (no display_name)")

minimal = v3.create_resource(path=xlsx_v1, resource_type=ResourceTypeDto.MODEL)
track(minimal.resource_id)
verify_resource_metadata(
    v3, minimal.resource_id,
    name=xlsx_v1.name, display_name=None, external_identifier=None, version_name=None,
)
report_upload(
    ui_base, minimal.resource_id,
    name=xlsx_v1.name, display_name=None, external_identifier=None, version_name=None,
)

## 2 · Upload a resource with display_name

### What to check in the web app

Open the **Files** page (`UI:` link in the output). The new row should show the **display name** from this cell (for example `using-resources-recipe UAS Requirements (…)`), not only the underlying filename. Click that row to open the **file detail** page and confirm the same title appears at the top of the page.

In [ ]:
from istari_digital_client.v3.models import ResourceTypeDto

section("2 · Upload a resource with display_name")

display_name = f"{RECIPE_TAG} UAS Requirements ({run_id})"
resource_id = v3.create_resource(
    path=xlsx_v1,
    resource_type=ResourceTypeDto.MODEL,
    display_name=display_name,
    description=f"Cookbook upload demo {run_id}",
    version_name="v1",
).resource_id
state["primary_resource_id"] = resource_id
track(resource_id)
verify_resource_metadata(
    v3, resource_id,
    name=xlsx_v1.name, display_name=display_name, external_identifier=None, version_name="v1",
)
report_upload(
    ui_base, resource_id,
    name=xlsx_v1.name, display_name=display_name, external_identifier=None, version_name="v1",
)

## 3 · Upload with external id and version label

### What to check in the web app

Open the resource with the printed `UI:` link. On the **file detail** page, go to the **Versions** tab (the list of file revisions). Open the current revision’s details. You should see an **External ID** (or external identifier) and a **Version** label that match the `external_identifier` and `version_name` values printed by this cell — these are extra labels your systems can use to find the file, separate from the display name on the Files list.

In [ ]:
from istari_digital_client.v3.models import ResourceTypeDto

section("3 · Upload with external id and version label")

external_id, external_version = f"{RECIPE_TAG}-ext-{run_id}", f"{RECIPE_TAG}-ver-{run_id}"
external_display_name = f"{RECIPE_TAG} external ids ({run_id})"
external_resource_id = v3.create_resource(
    path=xlsx_v1,
    resource_type=ResourceTypeDto.MODEL,
    display_name=external_display_name,
    external_identifier=external_id,
    version_name=external_version,
).resource_id
state["external_resource_id"] = external_resource_id
track(external_resource_id)
verify_resource_metadata(
    v3, external_resource_id,
    name=xlsx_v1.name, display_name=external_display_name,
    external_identifier=external_id, version_name=external_version,
)
report_upload(
    ui_base, external_resource_id,
    name=xlsx_v1.name, display_name=external_display_name,
    external_identifier=external_id, version_name=external_version,
)

## 4 · Search and filter resources

### What to check in the web app

On the **Files** page, try the same kinds of lookups the code performs: search or filter by **name**, open a file by following its link (equivalent to `get_resource`), and—if you uploaded step 3—find a resource using its **external identifier** and **version** in the UI filters or search, if your tenant exposes them. The API calls in this cell mirror those operations; you do not have to match every filter in the UI, but the resources from earlier steps should still appear when you browse or search.

Also demonstrates `get_resource_revision`, lookup by `external_identifier`, and lookup by `external_identifier` + `version_name` (using the resource from step 3).

In [ ]:
from istari_digital_client.v3.models import ArchiveStatus


def assert_in_page(page, resource_id: str) -> None:
    assert any(r.resource_id == resource_id for r in page.items)


section("4 · Search and filter resources")

# Standalone: resource_id = PLACEHOLDER_ID; optional external_resource_id, display_name, …
if "resource_id" not in dir():
    resource_id = PLACEHOLDER_ID
external_resource_id = external_resource_id if "external_resource_id" in dir() else resource_id

detail = v3.get_resource(resource_id=resource_id)
ext = detail if external_resource_id == resource_id else v3.get_resource(resource_id=external_resource_id)

if "display_name" not in dir():
    display_name = detail.display_name
if "description_text" not in dir():
    description_text = detail.description
if "version_label" not in dir():
    version_label = detail.version_name
external_id = external_id if "external_id" in dir() else ext.external_identifier
external_version = external_version if "external_version" in dir() else ext.version_name

by_name = v3.list_resources(name=[detail.name], size=50, include_total=True)
assert by_name.total and resource_id in {r.resource_id for r in by_name.items}
print(f"name={detail.name!r} → {by_name.total} match(es)")

assert detail.display_name == display_name and detail.version_name == version_label
print(f"get_resource → display_name={detail.display_name!r}")

revision = v3.get_resource_revision(resource_id=resource_id, revision_id=detail.file_revision_id)
assert revision.version_name == version_label
print(f"get_resource_revision → version_name={revision.version_name!r}")

if external_id:
    assert_in_page(v3.list_resources(external_identifier=[external_id]), external_resource_id)
    print(f"external_identifier={external_id!r} → {external_resource_id}")
    if external_version:
        page = v3.list_resources(external_identifier=[external_id], version_name=[external_version])
        match = next(r for r in page.items if r.resource_id == external_resource_id)
        assert match.external_identifier == external_id and match.version_name == external_version
        print(f"external_identifier+version → {match.resource_id}")

current_user = client.get_current_user()
print(f"current_user → {current_user.email!r}")

if description_text:
    assert_in_page(v3.list_resources(description=[description_text]), resource_id)
if version_label:
    assert_in_page(v3.list_resources(version_name=[version_label]), resource_id)
assert_in_page(v3.list_resources(created_by_id=[detail.created_by_id]), resource_id)
assert_in_page(v3.list_resources(archive_status=ArchiveStatus.ACTIVE), resource_id)
assert not detail.archived
print(f"list_resources filters include {resource_id}")

## 5 · Browse resources you can access

### What to check in the web app

Open the **Files** page. You should see a table (or list) of every **model** file your account can access that is not archived. The cell prints a few sample rows, then confirms it walked through **all** pages of results in the API (cursor pagination). In the UI you typically scroll or page through the same list; the total count printed here should be in the same ballpark as what the Files page reports for active models.

In [ ]:
from istari_digital_client.v3.models import ArchiveStatus

section("5 · Browse resources you can access")

cursor, seen, preview, total = None, set(), 0, None
while True:
    page = v3.list_resources(
        type_name=["model"], archive_status=ArchiveStatus.ACTIVE,
        size=100, cursor=cursor, include_total=True,
    )
    if total is None:
        total = page.total
        print(f"Active models total: {total}")
    for r in page.items:
        seen.add(r.resource_id)
        if preview < 5:
            print(f"  {r.display_name or r.name}  id={r.resource_id}")
            preview += 1
    cursor = page.next_page
    if not cursor:
        break

assert len(seen) == total
if "resource_id" in dir():
    assert resource_id in seen
print(f"Iterated {len(seen)} active model(s)")

## 6 · Add a comment

### What to check in the web app

Open the primary resource from step 2 using the `UI:` link in the output. On the **file detail** page, select the **Comments** tab. You should see a new comment whose text includes `using-resources-recipe` (the tag this notebook adds). Comments are notes attached to the file for reviewers; they do not change the file content.


In [ ]:
section("6 · Add a comment")

if "resource_id" not in dir():
    resource_id = PLACEHOLDER_ID

path = Path(tempfile.gettempdir()) / f"{RECIPE_TAG}-comment-{run_id}.txt"
path.write_text(f"Review note from {RECIPE_TAG} at {run_id}\n")
try:
    comment = v3.create_comment(resource_id=resource_id, path=path)
    state["comment_ids"].append(comment.id)
    assert RECIPE_TAG in v3.get_content(comment).decode()
    print(f"comment id={comment.id}  UI: {ui_base}/files/{resource_id}")
finally:
    path.unlink(missing_ok=True)

## 7 · View resource details

### What to check in the web app

Open the same resource (`UI:` link from step 2 or the printed output). On the **file detail** page, verify:

1. **Header** — resource id, file name, and current revision match the printed output.
2. **Versions** tab — at least one revision is listed (more after step 8).
3. **Comments** tab — the comment from step 6 is still visible.
4. **Share** (or sharing) panel — lists users who have access and their roles (for example Owner, Editor, Viewer). The cell prints the same sharing and permission information from the API for comparison.

This cell is **read-only**: it reports how the system is currently configured and does not modify the resource or its permissions.

### What `list_access` returns

`client.list_access` (`GET /api/v2/access/{resource_type}/{resource_id}`) returns the public `AccessRelationship` rows for the resource. Per the client:

- `AccessRelationship` is documented as *"PUBLIC: An access relationship that can be viewed/modified by a user of Istari."*
- The only `AccessSubjectType` exposed is `USER`, so every row is a single-user grant.
- The available `AccessRelation` values are `viewer`, `editor`, `owner`, `administrator`, `executor`, and `upstreamremoteowner`.
- The `resource_type` arg is derived from `detail.resource_type` so the same code works for `model`, `artifact`, or `file` resources.

If a caller can read the resource but no row for them appears in this list, the Python client doesn't expose any additional API to explain how their access was granted — so this notebook does not speculate. Inspect the platform UI's Share panel and any control-tag configuration for the resource if you need to investigate further.

### Control tags (`detail.control_tags`)

The cell also prints the resource's control tags. Per the `ControlTag` docstring: *"A control (essentially a tag) that is assigned to resource, files, and users. To have access to a resource or file that has one or more controls assigned, the user must have been assigned all the controls applied to the item."* — control tags are a **restriction** on top of the access grants above, not a grant in their own right.

### Effective permissions (`list_resource_type_permissions`)

The Share panel roles above are `AccessRelation` values (`viewer`, `editor`, `owner`, `administrator`). Separately, the authorization layer exposes action permissions via the `Permission` enum. For a model/artifact/file, the client and v3 API only document two that map to everyday UI capability:

| `Permission` | Where documented | UI meaning |
|---|---|---|
| `view` | v3 `list_resources` / `get_resource` responses refer to VIEW permission; `test_access.py` uses `Permission.VIEW` with `list_resource_type_permissions` | Can open and read the resource |
| `edit` | Same endpoint; no dedicated test for models, but this is the edit counterpart to `view` | Can modify the resource |

Other `Permission` values in the enum (`manage`, `access`, `archive`, `execute`, `access_view_manage`, `access_edit_manage`, `access_administrate_manage`, …) are not referenced by the resources API or resource tests in istari-python-client 10.10.0, so this notebook does not query them.

`client.list_resource_type_permissions(subject_type, subject_id, resource_type, permission)` (`GET /api/v2/access/{subject_type}/{subject_id}/{resource_type}?permission=…`) returns every resource of `resource_type` on which the subject holds the requested permission. The cell checks whether `resource_id` appears in those rows — surfacing effective access even when the caller has no row in `list_access`. One HTTP call per permission; failures are reported and skipped so one bad call does not abort the cell.


In [ ]:
from istari_digital_client.v2.models import (
    AccessResourceType,
    Permission,
    PermissionResourceType,
    PermissionSubjectType,
)


def _enum_value(value) -> str:
    return value.value if hasattr(value, "value") else value


def to_access_type(resource_type) -> AccessResourceType:
    return AccessResourceType(_enum_value(resource_type))


def to_permission_type(resource_type) -> PermissionResourceType:
    return PermissionResourceType(_enum_value(resource_type))


def print_grant(entry, current_user_id: str | None = None) -> None:
    info = entry.subject_info
    who = (info.username or info.email or entry.subject_id) if info else entry.subject_id
    tags = []
    if current_user_id and entry.subject_id == current_user_id:
        tags.append("(you)")
    if info and info.cross_tenant_user:
        tags.append("[cross-tenant]")
    suffix = f" {' '.join(tags)}" if tags else ""
    print(f"  - {entry.relation.value:<14} {entry.subject_type.value}={who}{suffix}")


def describe_access(client, user_id: str, resource_id: str, grants, resource_type) -> dict:
    explicit = next((g.relation.value for g in grants if g.subject_id == user_id), None)
    prt = to_permission_type(resource_type)

    def effective(perm: Permission) -> bool | None:
        try:
            rows = client.list_resource_type_permissions(
                subject_type=PermissionSubjectType.USER,
                subject_id=user_id,
                resource_type=prt,
                permission=perm,
            )
        except Exception:
            return None
        return any(r.resource_id == resource_id for r in rows)

    return {
        "explicit_share_role": explicit,
        "effective_view": effective(Permission.VIEW),
        "effective_edit": effective(Permission.EDIT),
    }


section("7 · View resource details")

if "resource_id" not in dir():
    resource_id = PLACEHOLDER_ID

detail = v3.get_resource(resource_id=resource_id)
current_user = current_user if "current_user" in dir() else client.get_current_user()

revisions = v3.list_resource_revisions(resource_id=resource_id, include_total=True)
comments_page = v3.list_comments(resource_id=resource_id, include_total=True)
assert detail.file_id and detail.file_revision_id
assert revisions.total and revisions.total >= 1
assert comments_page.total and comments_page.total >= 1
if "comment" in dir():
    assert any(c.id == comment.id for c in comments_page.items)

print(f"resource_id={detail.resource_id}")
print(f"file_id={detail.file_id}  revision_id={detail.file_revision_id}")
print(f"Revisions: {revisions.total}  Comments: {comments_page.total}")

tags = [t.name for t in (detail.control_tags or []) if t and t.name]
print(f"Control tags: {', '.join(tags) if tags else '(none)'}")

# v2 list_access expects AccessResourceType; v3 uses ResourceTypeDto (same wire strings).
access_list = client.list_access(resource_type=to_access_type(detail.resource_type), resource_id=resource_id)
print(f"Access relationships ({len(access_list)} row(s)):")
for entry in access_list:
    print_grant(entry, current_user.id)

summary = describe_access(client, current_user.id, resource_id, access_list, detail.resource_type)
label = current_user.email or current_user.id
print(f"Access summary for {label}:")
role = summary["explicit_share_role"]
print(f"  explicit Share role: {role!r}" if role else "  explicit Share role: (none)")
for name, key in (("view", "effective_view"), ("edit", "effective_edit")):
    val = summary[key]
    print(f"  effective {name:<4} {'error' if val is None else 'YES' if val else 'no'}")

## 8 · Upload a new version

### What to check in the web app

Open the primary resource and go to the **Versions** tab. You should now see **at least two** revisions: the original upload and a new row for version `v2` (with the updated display name suffix). The newest revision is the file content this cell uploaded from disk.

In [ ]:
section("8 · Upload a new version")

if "resource_id" not in dir():
    resource_id = PLACEHOLDER_ID

detail = v3.get_resource(resource_id=resource_id)
display_name = display_name if "display_name" in dir() else (detail.display_name or detail.name)

revision_v2 = v3.create_resource_revision(
    resource_id=resource_id,
    path=xlsx_v2 if xlsx_v2.is_file() else xlsx_v1,
    description=f"Second revision {run_id}",
    version_name="v2",
    display_name=f"{display_name} (v2)",
)
state["revision_v2_id"] = revision_v2.file_revision_id
assert revision_v2.owning_entity_id == resource_id
assert revision_v2.file_revision_id != detail.file_revision_id
assert v3.list_resource_revisions(resource_id=resource_id, include_total=True).total >= 2
print(f"New revision_id={revision_v2.file_revision_id}")

## 9 · Compare resource versions

### What to check in the web app

On the **Versions** tab, select the first and second revisions and use the platform’s **Compare** action (wording may vary by tenant). The UI highlights differences between the two files. This notebook compares the two files in Python by downloading both revisions and checking whether the bytes match; there is no separate “compare” API in istari-digital-client 10.10.0, so the UI is the place to see a visual diff.

In [ ]:
section("9 · Compare resource versions")

if "resource_id" not in dir():
    resource_id = PLACEHOLDER_ID

if "rev_v1_id" not in dir():
    rev_v1_id = detail.file_revision_id if "detail" in dir() else None
if "rev_v2_id" not in dir():
    rev_v2_id = revision_v2.file_revision_id if "revision_v2" in dir() else None

if not rev_v1_id or not rev_v2_id:
    page = v3.list_resource_revisions(resource_id=resource_id, include_total=True)
    assert page.total >= 2, "Need 2+ revisions — run §8 or pick another resource_id"
    by_time = sorted(page.items, key=lambda r: r.created)
    rev_v1_id = rev_v1_id or by_time[0].file_revision_id
    rev_v2_id = rev_v2_id or by_time[-1].file_revision_id

rev_v1 = v3.get_resource_revision(resource_id=resource_id, revision_id=rev_v1_id)
rev_v2 = v3.get_resource_revision(resource_id=resource_id, revision_id=rev_v2_id)
c1, c2 = v3.get_content(rev_v1), v3.get_content(rev_v2)
print(f"v1 bytes={len(c1)}  v2 bytes={len(c2)}  identical={c1 == c2}")
if c1 != c2:
    assert rev_v1.content_token and rev_v2.content_token
    assert rev_v1.content_token.sha != rev_v2.content_token.sha
print(f"Compare in UI: {ui_base}/files/{resource_id} (Versions → Compare)")

## 10 · Download resource content

### What to check in the web app

On the **file detail** page (or on a specific revision), use the **Download** button to save a copy to your computer. This cell does the same thing via the API: it writes the current file bytes to a path under your system temp directory and prints that path. The downloaded size should match the `size` reported for the resource when the platform provides it.

In [ ]:
section("10 · Download resource content")

if "resource_id" not in dir():
    resource_id = PLACEHOLDER_ID

detail = v3.get_resource(resource_id=resource_id)
dest = Path(tempfile.gettempdir()) / f"{RECIPE_TAG}-{run_id}{Path(detail.name or '').suffix or '.bin'}"
dest.write_bytes(v3.get_content(detail))
size = dest.stat().st_size
assert size == detail.size if detail.size is not None else size > 0
state["download_path"] = str(dest)
print(f"Wrote {dest} ({size} bytes)")

## 11 · Archive a resource (throwaway)

### What to check in the web app

This cell creates a **separate** file only to demonstrate archiving. After it runs, that file should **disappear from the default active Files list** (or show as archived, depending on your tenant’s UI). The main demo resource from step 2 is **not** archived here—it stays active so you can keep reviewing it until the cleanup cell.

In [ ]:
from istari_digital_client.v3.models import ArchiveStatus, ResourceTypeDto

section("11 · Archive a resource (throwaway)")

name = f"{RECIPE_TAG} archive demo ({run_id})"
rid = v3.create_resource(
    path=xlsx_v1, resource_type=ResourceTypeDto.FILE, display_name=name,
).resource_id
state["archive_demo_resource_id"] = rid
track(rid)
verify_resource_metadata(v3, rid, name=xlsx_v1.name, display_name=name, external_identifier=None, version_name=None)

v3.archive_resource(resource_id=rid)
assert v3.get_resource(resource_id=rid).archived
assert len(v3.list_resources(resource_id=[rid], archive_status=ArchiveStatus.ARCHIVED).items) == 1
print(f"Archived throwaway resource_id={rid}")

## 12 · Cleanup — archive demo resources

### What to check in the web app

After you finish reviewing the platform, run the code cell below in the **same Jupyter kernel session** as the rest of this notebook (so the in-memory `state` still lists every resource id created here). The cell archives each of those resources and deletes the temp file from step 10. When cleanup succeeds, the demo files from steps 1–3 and step 2’s primary resource should no longer appear among **active** files on the Files page.

In [ ]:
from istari_digital_client.v3.models import ArchiveStatus

section("12 · Cleanup — archive demo resources")

# If the kernel restarted: state = {"run_id": "manual", "resource_ids": [...], "download_path": None}
if "state" not in dir():
    raise RuntimeError("Run Setup + demo cells first, or set state = {...}")

ids = list(dict.fromkeys(state.get("resource_ids", [])))
print(f"Cleaning run_id={state.get('run_id', '?')} ({len(ids)} resource(s))")

for rid in ids:
    try:
        current = v3.get_resource(resource_id=rid)
    except Exception as exc:
        print(f"  skip {rid}: {exc}")
        continue
    if current.archived:
        print(f"  already archived: {rid}")
        continue
    v3.archive_resource(resource_id=rid)
    assert v3.get_resource(resource_id=rid).archived
    assert not v3.list_resources(resource_id=[rid], archive_status=ArchiveStatus.ACTIVE).items
    print(f"  archived: {rid}")

download_path = state.get("download_path")
if download_path and Path(download_path).is_file():
    Path(download_path).unlink()
    print(f"  removed download: {download_path}")

print("Cleanup complete — you can rerun this notebook from the top.")

## Done

All demo resources for this run have been archived and the local download removed.